In [4]:
import time, os, json
from pyspark.sql import SparkSession

# =====================================================
# Sessão Spark
# =====================================================
spark = (
    SparkSession.builder
    .appName("Benchmark-Formats")
    .master("local[*]")
    .config("spark.executor.memory", "1g")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.committer.name", "directory")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

# =====================================================
# Dataset base (local CSV)
# =====================================================
schema = "raceId INT, driverId INT, lap INT, position INT, time STRING, milliseconds INT"
df = spark.read.csv("/home/jovyan/data/lap_times.csv", header=True, schema=schema)
print("Linhas carregadas:", df.count())

# =====================================================
# Funções auxiliares
# =====================================================
def medir_tempo(op, label):
    inicio = time.time()
    result = op()
    fim = time.time()
    print(f"{label}: {fim - inicio:.2f}s")
    return fim - inicio, result

def medir_leitura(fmt, path):
    inicio = time.time()
    if fmt == "csv":
        df = spark.read.option("header", True).csv(path)
    elif fmt == "json":
        df = spark.read.json(path)
    elif fmt == "parquet":
        df = spark.read.parquet(path)
    elif fmt == "avro":
        df = spark.read.format("avro").option("ignoreExtension", "true").load(path)
    elif fmt == "orc":
        df = spark.read.orc(path)
    linhas = df.count()
    fim = time.time()
    return fim - inicio, linhas

# =====================================================
# Escrita em diferentes formatos no MinIO
# =====================================================
base = "s3a://datalake-bronze/lap_times_"
formats = ["csv", "json", "parquet", "avro", "orc"]
tempos_escrita = {}
for fmt in formats:
    path = base + fmt
    print(f"Gravando {fmt.upper()} -> {path}")
    inicio = time.time()
    if fmt == "csv":
        df.write.mode("overwrite").option("header", True).csv(path)
    elif fmt == "json":
        df.write.mode("overwrite").json(path)
    elif fmt == "parquet":
        df.write.mode("overwrite").parquet(path)
    elif fmt == "avro":
        df.write.mode("overwrite").format("avro").save(path)
    elif fmt == "orc":
        df.write.mode("overwrite").orc(path)
    fim = time.time()
    tempos_escrita[fmt] = round(fim - inicio, 2)
    print(f"→ Tempo de escrita {fmt.upper()}: {tempos_escrita[fmt]}s")

# =====================================================
# Verifica tamanhos com mc (precisa alias 'local')
# =====================================================
!mc alias set local http://minio:9000 minioadmin minioadmin >/dev/null
tamanhos = {}
for fmt in formats:
    info = os.popen(f"mc du --json local/datalake/lap_times_{fmt}/").read()
    try:
        tamanho = float(json.loads(info)["size"]) / (1024 * 1024)
    except:
        tamanho = 0
    tamanhos[fmt] = round(tamanho, 2)

# =====================================================
# Leitura e benchmark
# =====================================================
tempos_leitura = {}
linhas_lidas = {}
for fmt in formats:
    path = base + fmt
    try:
        tempo, linhas = medir_leitura(fmt, path)
        tempos_leitura[fmt] = round(tempo, 2)
        linhas_lidas[fmt] = linhas
        print(f"{fmt.upper()}: {linhas} linhas em {tempo:.2f}s")
    except Exception as e:
        tempos_leitura[fmt] = None
        linhas_lidas[fmt] = 0
        print(f"Erro ao ler {fmt}: {e}")

# =====================================================
# Resultado consolidado
# =====================================================
print("\n=== RESULTADO FINAL ===")
print(f"{'Formato':<10} {'Tamanho (MB)':>12} {'Escrita (s)':>12} {'Leitura (s)':>12} {'Linhas':>10}")
for fmt in formats:
    print(f"{fmt.upper():<10} {tamanhos[fmt]:>12.2f} {tempos_escrita[fmt]:>12.2f} {tempos_leitura[fmt] or 0:>12.2f} {linhas_lidas[fmt]:>10}")


Linhas carregadas: 551742
Gravando CSV -> s3a://datalake-bronze/lap_times_csv


Py4JJavaError: An error occurred while calling o102.csv.
: org.apache.hadoop.fs.s3a.UnknownStoreException: `s3a://datalake-bronze/lap_times_csv': getFileStatus on s3a://datalake-bronze/lap_times_csv: com.amazonaws.services.s3.model.AmazonS3Exception: The specified bucket does not exist (Service: Amazon S3; Status Code: 404; Error Code: NoSuchBucket; Request ID: 18DC41F32AC845DB; S3 Extended Request ID: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8; Proxy: null), S3 Extended Request ID: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8:NoSuchBucket: The specified bucket does not exist (Service: Amazon S3; Status Code: 404; Error Code: NoSuchBucket; Request ID: 18DC41F32AC845DB; S3 Extended Request ID: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8; Proxy: null)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:263)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.